# MovieLens 20M — Data Exploration (Stage 1)

Explore `ratings.csv`, `movies.csv`, and `tags.csv` before locking the Elasticsearch schema.

**Raw data path:** `../data/raw/`

**Kernel:** use project venv — `Python (bigData)` or `.venv/Scripts/python.exe`

Run all cells top-to-bottom. Full dataset (~20M ratings) may take a few minutes to load.

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_RAW = Path("../data/raw")
RATINGS_PATH = DATA_RAW / "ratings.csv"
MOVIES_PATH = DATA_RAW / "movies.csv"
TAGS_PATH = DATA_RAW / "tags.csv"

for path in (RATINGS_PATH, MOVIES_PATH, TAGS_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

print("Data files found.")

Data files found.


In [2]:
ratings = pd.read_csv(RATINGS_PATH)
movies = pd.read_csv(MOVIES_PATH)
tags = pd.read_csv(TAGS_PATH)

print(f"ratings: {len(ratings):,} rows")
print(f"movies:  {len(movies):,} rows")
print(f"tags:    {len(tags):,} rows")

ratings: 20,000,263 rows
movies:  27,278 rows
tags:    465,564 rows


## 1. Ratings analysis (`ratings.csv`)

Columns: `userId`, `movieId`, `rating`, `timestamp`

In [3]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,2,3.50,2005-04-02 23:53:47
1,1,29,3.50,2005-04-02 23:31:16
2,1,32,3.50,2005-04-02 23:33:39
3,1,47,3.50,2005-04-02 23:32:07
4,1,50,3.50,2005-04-02 23:29:40


In [4]:
ratings.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000263 entries, 0 to 20000262
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int64  
 1   movieId    int64  
 2   rating     float64
 3   timestamp  str    
dtypes: float64(1), int64(2), str(1)
memory usage: 972.8 MB


In [5]:
print("Missing values:")
print(ratings.isna().sum())

duplicate_rows = ratings.duplicated().sum()
print(f"\nDuplicate rows: {duplicate_rows:,}")

print(f"\nUnique users:  {ratings['userId'].nunique():,}")
print(f"Unique movies: {ratings['movieId'].nunique():,}")

Missing values:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Duplicate rows: 0

Unique users:  138,493
Unique movies: 26,744


In [6]:
print("Rating distribution:")
print(ratings["rating"].describe())
print("\nValue counts:")
print(ratings["rating"].value_counts().sort_index())

valid_half_steps = ratings["rating"].mod(0.5).eq(0).all()
in_range = ratings["rating"].between(0.5, 5.0).all()
print(f"\nAll ratings in [0.5, 5.0]: {in_range}")
print(f"All ratings on 0.5 steps:   {valid_half_steps}")

Rating distribution:
count   20,000,263.00
mean             3.53
std              1.05
min              0.50
25%              3.00
50%              3.50
75%              4.00
max              5.00
Name: rating, dtype: float64

Value counts:
rating
0.50     239125
1.00     680732
1.50     279252
2.00    1430997
2.50     883398
3.00    4291193
3.50    2200156
4.00    5561926
4.50    1534824
5.00    2898660
Name: count, dtype: int64

All ratings in [0.5, 5.0]: True
All ratings on 0.5 steps:   True


In [7]:
ratings["timestamp"] = pd.to_datetime(ratings["timestamp"])
ratings["rating_year"] = ratings["timestamp"].dt.year

print(f"Timestamp range: {ratings['timestamp'].min()} → {ratings['timestamp'].max()}")
print("\nRatings by rating_year (top 10):")
print(ratings["rating_year"].value_counts().sort_index().tail(10))

ratings_per_movie = ratings.groupby("movieId").size()
ratings_per_user = ratings.groupby("userId").size()

print("\nRatings per movie:")
print(ratings_per_movie.describe())
print("\nRatings per user:")
print(ratings_per_user.describe())

Timestamp range: 1995-01-09 11:46:44 → 2015-03-31 06:40:02

Ratings by rating_year (top 10):
rating_year
2006    1171836
2007    1053430
2008    1158777
2009     930036
2010     903691
2011     766366
2012     731389
2013     599327
2014     562888
2015     283886
Name: count, dtype: int64

Ratings per movie:
count   26,744.00
mean       747.84
std      3,085.82
min          1.00
25%          3.00
50%         18.00
75%        205.00
max     67,310.00
dtype: float64

Ratings per user:
count   138,493.00
mean        144.41
std         230.27
min          20.00
25%          35.00
50%          68.00
75%         155.00
max       9,254.00
dtype: float64


## 2. Movies analysis (`movies.csv`)

Columns: `movieId`, `title`, `genres`

In [8]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [9]:
print("Missing values:")
print(movies.isna().sum())

dup_movie_ids = movies["movieId"].duplicated().sum()
print(f"\nDuplicate movieId values: {dup_movie_ids:,}")
print(f"Total movies: {len(movies):,}")

Missing values:
movieId    0
title      0
genres     0
dtype: int64

Duplicate movieId values: 0
Total movies: 27,278


In [10]:
TITLE_YEAR_PATTERN = re.compile(r"\((\d{4})\)\s*$")


def extract_release_year(title: str) -> int | None:
    match = TITLE_YEAR_PATTERN.search(str(title))
    return int(match.group(1)) if match else None


def parse_genres(genres: str) -> list[str]:
    if pd.isna(genres) or genres == "(no genres listed)":
        return []
    return [g.strip() for g in str(genres).split("|") if g.strip()]


movies["release_year"] = movies["title"].map(extract_release_year)
movies["genres_list"] = movies["genres"].map(parse_genres)
movies["genre_count"] = movies["genres_list"].map(len)

parsed_years = movies["release_year"].notna().sum()
no_genre = (movies["genres"] == "(no genres listed)").sum()

print(f"Titles with parsed release year: {parsed_years:,} ({parsed_years / len(movies):.1%})")
print(f"Movies with (no genres listed):  {no_genre:,}")
print(f"\nRelease year range: {movies['release_year'].min():.0f} → {movies['release_year'].max():.0f}")
print("\nSample parsed movies:")
movies[["movieId", "title", "release_year", "genres_list"]].head(10)

Titles with parsed release year: 27,252 (99.9%)
Movies with (no genres listed):  246

Release year range: 1891 → 2015

Sample parsed movies:


,movieId,title,release_year,genres_list
0,1,Toy Story (1995),"1,995.00","[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"1,995.00","[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"1,995.00","[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"1,995.00","[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),"1,995.00",[Comedy]
5,6,Heat (1995),"1,995.00","[Action, Crime, Thriller]"
6,7,Sabrina (1995),"1,995.00","[Comedy, Romance]"
7,8,Tom and Huck (1995),"1,995.00","[Adventure, Children]"
8,9,Sudden Death (1995),"1,995.00",[Action]
9,10,GoldenEye (1995),"1,995.00","[Action, Adventure, Thriller]"


In [11]:
genre_counts = (
    movies.explode("genres_list")
    .dropna(subset=["genres_list"])
    .groupby("genres_list")
    .size()
    .sort_values(ascending=False)
)

print("Movies per genre (top 15):")
print(genre_counts.head(15))

Movies per genre (top 15):
genres_list
Drama          13344
Comedy          8374
Thriller        4178
Romance         4127
Action          3520
Crime           2939
Horror          2611
Documentary     2471
Adventure       2329
Sci-Fi          1743
Mystery         1514
Fantasy         1412
War             1194
Children        1139
Musical         1036
dtype: int64


## 3. Tags analysis (`tags.csv`)

Columns: `userId`, `movieId`, `tag`, `timestamp`

In [12]:
tags.head()

,userId,movieId,tag,timestamp
0,18,4141,Mark Waters,2009-04-24 18:19:40
1,65,208,dark hero,2013-05-10 01:41:18
2,65,353,dark hero,2013-05-10 01:41:19
3,65,521,noir thriller,2013-05-10 01:39:43
4,65,592,dark hero,2013-05-10 01:41:18


In [13]:
print("Missing values:")
print(tags.isna().sum())

print(f"\nTag records:   {len(tags):,}")
print(f"Unique tags:   {tags['tag'].nunique():,}")
print(f"Unique movies: {tags['movieId'].nunique():,}")
print(f"Unique users:  {tags['userId'].nunique():,}")

duplicate_tag_rows = tags.duplicated().sum()
print(f"\nDuplicate rows (exact): {duplicate_tag_rows:,}")

Missing values:
userId        0
movieId       0
tag          16
timestamp     0
dtype: int64

Tag records:   465,564
Unique tags:   38,643
Unique movies: 19,545
Unique users:  7,801

Duplicate rows (exact): 0


In [14]:
tags["tag_normalized"] = tags["tag"].str.strip().str.lower()

raw_unique = tags["tag"].nunique()
normalized_unique = tags["tag_normalized"].nunique()
print(f"Unique tags (raw):        {raw_unique:,}")
print(f"Unique tags (normalized): {normalized_unique:,}")
print(f"Reduction after lowercasing: {raw_unique - normalized_unique:,}")

print("\nMost common tags (normalized, top 20):")
print(tags["tag_normalized"].value_counts().head(20))

Unique tags (raw):        38,643
Unique tags (normalized): 35,170
Reduction after lowercasing: 3,473

Most common tags (normalized, top 20):
tag_normalized
sci-fi              3576
based on a book     3307
atmospheric         3169
comedy              3078
action              3068
nudity (topless)    2646
surreal             2528
twist ending        2367
bd-r                2334
funny               2253
quirky              2034
dystopia            2015
classic             1971
stylized            1944
dark comedy         1910
romance             1874
fantasy             1850
psychology          1763
time travel         1572
disturbing          1524
Name: count, dtype: int64


In [15]:
tags["timestamp"] = pd.to_datetime(tags["timestamp"])
tags_per_movie = tags.groupby("movieId").size()

print(f"Tag timestamp range: {tags['timestamp'].min()} → {tags['timestamp'].max()}")
print("\nTags per movie:")
print(tags_per_movie.describe())

Tag timestamp range: 2005-12-24 13:00:10 → 2015-03-31 03:09:12

Tags per movie:
count   19,545.00
mean        23.82
std         70.07
min          1.00
25%          2.00
50%          5.00
75%         16.00
max      1,994.00
dtype: float64


## 4. Cross-file checks

Verify joins and orphan references across files.

In [16]:
movie_ids = set(movies["movieId"])
rated_movie_ids = set(ratings["movieId"].unique())
tagged_movie_ids = set(tags["movieId"].unique())

movies_no_ratings = movie_ids - rated_movie_ids
movies_no_tags = movie_ids - tagged_movie_ids
ratings_unknown_movies = rated_movie_ids - movie_ids
tags_unknown_movies = tagged_movie_ids - movie_ids

print(f"Movies in catalog:              {len(movie_ids):,}")
print(f"Movies with at least one rating: {len(rated_movie_ids):,}")
print(f"Movies with at least one tag:    {len(tagged_movie_ids):,}")
print(f"\nMovies with no ratings:          {len(movies_no_ratings):,}")
print(f"Movies with no tags:             {len(movies_no_tags):,}")
print(f"Ratings referencing unknown movieId: {len(ratings_unknown_movies):,}")
print(f"Tags referencing unknown movieId:    {len(tags_unknown_movies):,}")

Movies in catalog:              27,278
Movies with at least one rating: 26,744
Movies with at least one tag:    19,545

Movies with no ratings:          534
Movies with no tags:             7,733
Ratings referencing unknown movieId: 0
Tags referencing unknown movieId:    0


In [17]:
ratings_with_movies = ratings.merge(movies, on="movieId", how="inner")
tags_with_movies = tags.merge(movies, on="movieId", how="inner")

print(f"Ratings joined to movies: {len(ratings_with_movies):,} / {len(ratings):,}")
print(f"Tags joined to movies:    {len(tags_with_movies):,} / {len(tags):,}")

if ratings_unknown_movies:
    print(f"\nSample unknown movieIds in ratings: {sorted(ratings_unknown_movies)[:10]}")
if tags_unknown_movies:
    print(f"Sample unknown movieIds in tags:    {sorted(tags_unknown_movies)[:10]}")

Ratings joined to movies: 20,000,263 / 20,000,263
Tags joined to movies:    465,564 / 465,564


## 5. Data quality summary

Key findings for the team before Stage 2 (schema lock).

In [18]:
summary = {
    "ratings_rows": len(ratings),
    "ratings_users": ratings["userId"].nunique(),
    "ratings_movies": ratings["movieId"].nunique(),
    "ratings_missing": int(ratings.isna().sum().sum()),
    "ratings_duplicates": int(ratings.duplicated().sum()),
    "rating_min": float(ratings["rating"].min()),
    "rating_max": float(ratings["rating"].max()),
    "rating_year_min": int(ratings["rating_year"].min()),
    "rating_year_max": int(ratings["rating_year"].max()),
    "movies_rows": len(movies),
    "movies_missing": int(movies.isna().sum().sum()),
    "movies_duplicate_ids": int(movies["movieId"].duplicated().sum()),
    "movies_no_genres_listed": int((movies["genres"] == "(no genres listed)").sum()),
    "movies_with_release_year": int(movies["release_year"].notna().sum()),
    "tags_rows": len(tags),
    "tags_unique_raw": int(tags["tag"].nunique()),
    "tags_unique_normalized": int(tags["tag_normalized"].nunique()),
    "movies_no_ratings": len(movies_no_ratings),
    "movies_no_tags": len(movies_no_tags),
    "ratings_orphan_movie_ids": len(ratings_unknown_movies),
    "tags_orphan_movie_ids": len(tags_unknown_movies),
}

summary_df = pd.Series(summary, name="value").to_frame()
summary_df

,value
ratings_rows,"20,000,263.00"
ratings_users,"138,493.00"
ratings_movies,"26,744.00"
ratings_missing,0.00
ratings_duplicates,0.00
...,...
tags_unique_normalized,"35,170.00"
movies_no_ratings,534.00
movies_no_tags,"7,733.00"
ratings_orphan_movie_ids,0.00


### Takeaways for Elasticsearch schema (Stage 2)

1. **`rating_year`** — derive from rating `timestamp` (calendar year of when the rating was submitted).
2. **`release_year`** — parse from movie title pattern `(YYYY)` at end of string; not all titles may parse.
3. **`genres`** — pipe-delimited in CSV; parse to keyword array; handle `(no genres listed)`.
4. **`tags`** — free text; normalize to lowercase for aggregation and search.
5. **Joins** — `movieId` is the primary key across all three files; check orphan counts above.
6. **Rating values** — expect 0.5–5.0 in half-star steps; validate in ETL.

See also: `docs/data_quality_summary.md`